## PS1 - rescue robot pathfinding

grid has S, G, `.` free, `#` obstacle. need BFS and DFS from S to G, moves cost 1, successor order is UP RIGHT DOWN LEFT.

In [1]:
sample1 = '''5 5\nS..#.\n#..#.\n.....\n.##..\n....G'''
print(sample1)

5 5
S..#.
#..#.
.....
.##..
....G


S is at (0,0), G is at (4,4). manhattan distance is 8, and there don't look like any obstacles that force a detour, so shortest path should be exactly 8 moves if one exists.

In [2]:
from collections import deque

MOVES = [("UP", -1, 0), ("RIGHT", 0, 1), ("DOWN", 1, 0), ("LEFT", 0, -1)]

def parse_grid(text):
    lines = text.split(chr(10))
    r, c = map(int, lines[0].split())
    grid = lines[1:1+r]
    start = goal = None
    for i in range(r):
        for j in range(c):
            if grid[i][j] == 'S':
                start = (i, j)
            elif grid[i][j] == 'G':
                goal = (i, j)
    return grid, r, c, start, goal

grid, R, C, start, goal = parse_grid(sample1)
print(start, goal)

(0, 0) (4, 4)


good, matches the hand check. now build BFS step by step.

In [3]:
def in_bounds(r, c, R, C):
    return 0 <= r < R and 0 <= c < C

def build_path(parent, node):
    directions = []
    while node in parent:
        prev, d = parent[node]
        directions.append(d)
        node = prev
    directions.reverse()
    return directions

def bfs(grid, R, C, start, goal):
    visited = {start}
    parent = {}
    queue = deque([start])
    nodes_expanded = 0
    while queue:
        cur = queue.popleft()
        nodes_expanded += 1
        if cur == goal:
            return True, build_path(parent, cur), nodes_expanded
        r, c = cur
        for name, dr, dc in MOVES:
            nr, nc = r + dr, c + dc
            if in_bounds(nr, nc, R, C) and grid[nr][nc] != '#' and (nr, nc) not in visited:
                visited.add((nr, nc))
                parent[(nr, nc)] = (cur, name)
                queue.append((nr, nc))
    return False, [], nodes_expanded

found, path, nodes = bfs(grid, R, C, start, goal)
print(found, path, len(path), nodes)

True ['RIGHT', 'RIGHT', 'DOWN', 'DOWN', 'RIGHT', 'RIGHT', 'DOWN', 'DOWN'] 8 19


matches sample output: RIGHT RIGHT DOWN DOWN RIGHT RIGHT DOWN DOWN, 8 moves. now DFS, same successor order but a stack instead of a queue. push in reverse order so UP still comes out first.

In [4]:
def dfs(grid, R, C, start, goal):
    visited = {start}
    parent = {}
    stack = [start]
    nodes_expanded = 0
    while stack:
        cur = stack.pop()
        nodes_expanded += 1
        if cur == goal:
            return True, build_path(parent, cur), nodes_expanded
        r, c = cur
        for name, dr, dc in reversed(MOVES):
            nr, nc = r + dr, c + dc
            if in_bounds(nr, nc, R, C) and grid[nr][nc] != '#' and (nr, nc) not in visited:
                visited.add((nr, nc))
                parent[(nr, nc)] = (cur, name)
                stack.append((nr, nc))
    return False, [], nodes_expanded

found_d, path_d, nodes_d = dfs(grid, R, C, start, goal)
print(found_d, path_d, len(path_d), nodes_d)

True ['RIGHT', 'RIGHT', 'DOWN', 'DOWN', 'RIGHT', 'RIGHT', 'DOWN', 'DOWN'] 8 11


DFS found the same path here since this grid barely has branching, but nodes expanded is different from BFS. now check sample 2, which has no path.

In [5]:
sample2 = '''5 5\nS....\n####.\n....#\n.####\n....G'''
grid2, R2, C2, start2, goal2 = parse_grid(sample2)
print(bfs(grid2, R2, C2, start2, goal2))
print(dfs(grid2, R2, C2, start2, goal2))

(False, [], 6)
(False, [], 6)


both correctly report no path. now the full script that reads stdin and prints in the exact format the spec wants, run for real on sample 1.

In [6]:
import io, sys as _sys
_sys.stdin = io.StringIO(sample1)
__name__ = '__main__'
import sys
import time
from collections import deque

# moves must be generated in this order: UP RIGHT DOWN LEFT
MOVES = [("UP", -1, 0), ("RIGHT", 0, 1), ("DOWN", 1, 0), ("LEFT", 0, -1)]


def read_grid():
    data = sys.stdin.read().split("\n")
    r, c = map(int, data[0].split())
    grid = [data[i + 1] for i in range(r)]
    start = goal = None
    for i in range(r):
        for j in range(c):
            if grid[i][j] == "S":
                start = (i, j)
            elif grid[i][j] == "G":
                goal = (i, j)
    return grid, r, c, start, goal


def in_bounds(r, c, R, C):
    return 0 <= r < R and 0 <= c < C


def build_path(parent, node):
    directions = []
    while node in parent:
        prev, d = parent[node]
        directions.append(d)
        node = prev
    directions.reverse()
    return directions


def bfs(grid, R, C, start, goal):
    start_time = time.time()
    visited = {start}
    parent = {}
    queue = deque([start])
    nodes_expanded = 0

    while queue:
        cur = queue.popleft()
        nodes_expanded += 1
        if cur == goal:
            path = build_path(parent, cur)
            return True, path, nodes_expanded, time.time() - start_time

        r, c = cur
        for name, dr, dc in MOVES:
            nr, nc = r + dr, c + dc
            if in_bounds(nr, nc, R, C) and grid[nr][nc] != "#" and (nr, nc) not in visited:
                visited.add((nr, nc))
                parent[(nr, nc)] = (cur, name)
                queue.append((nr, nc))

    return False, [], nodes_expanded, time.time() - start_time


def dfs(grid, R, C, start, goal):
    start_time = time.time()
    visited = {start}
    parent = {}
    stack = [start]
    nodes_expanded = 0

    while stack:
        cur = stack.pop()
        nodes_expanded += 1
        if cur == goal:
            path = build_path(parent, cur)
            return True, path, nodes_expanded, time.time() - start_time

        r, c = cur
        # push in reverse so UP is popped first, keeping generation order UP RIGHT DOWN LEFT
        for name, dr, dc in reversed(MOVES):
            nr, nc = r + dr, c + dc
            if in_bounds(nr, nc, R, C) and grid[nr][nc] != "#" and (nr, nc) not in visited:
                visited.add((nr, nc))
                parent[(nr, nc)] = (cur, name)
                stack.append((nr, nc))

    return False, [], nodes_expanded, time.time() - start_time


def print_result(algo_name, found, path, nodes_expanded, exec_time):
    print(f"Algorithm: {algo_name}")
    if not found:
        print("Path Found: No")
        print(f"Nodes Expanded = {nodes_expanded}")
        print(f"Execution Time = {exec_time:.6f}")
        return
    print("Path Found: Yes")
    print("Path: " + " ".join(path))
    print(f"Number of Moves = {len(path)}")
    print(f"Nodes Expanded = {nodes_expanded}")
    print(f"Execution Time = {exec_time:.6f}")


def main():
    grid, R, C, start, goal = read_grid()

    bfs_found, bfs_path, bfs_nodes, bfs_time = bfs(grid, R, C, start, goal)
    print_result("BFS", bfs_found, bfs_path, bfs_nodes, bfs_time)

    print()

    dfs_found, dfs_path, dfs_nodes, dfs_time = dfs(grid, R, C, start, goal)
    print_result("DFS", dfs_found, dfs_path, dfs_nodes, dfs_time)

    print()
    print("Comparison:")
    if bfs_found and dfs_found:
        print(f"Path length -> BFS = {len(bfs_path)}, DFS = {len(dfs_path)}")
    print(f"Nodes expanded -> BFS = {bfs_nodes}, DFS = {dfs_nodes}")
    print(f"Execution time -> BFS = {bfs_time:.6f}, DFS = {dfs_time:.6f}")
    print("BFS is optimal (shortest path) since it explores level by level.")
    print("DFS is not guaranteed optimal, it just goes deep first.")


if __name__ == "__main__":
    main()


Algorithm: BFS
Path Found: Yes
Path: RIGHT RIGHT DOWN DOWN RIGHT RIGHT DOWN DOWN
Number of Moves = 8
Nodes Expanded = 19
Execution Time = 0.000989

Algorithm: DFS
Path Found: Yes
Path: RIGHT RIGHT DOWN DOWN RIGHT RIGHT DOWN DOWN
Number of Moves = 8
Nodes Expanded = 11
Execution Time = 0.000000

Comparison:
Path length -> BFS = 8, DFS = 8
Nodes expanded -> BFS = 19, DFS = 11
Execution time -> BFS = 0.000989, DFS = 0.000000
BFS is optimal (shortest path) since it explores level by level.
DFS is not guaranteed optimal, it just goes deep first.
